Till now we have see how to use Pipleine function from the transfomer library. But a big drawback is its inability to fine tune parameters. To Solve this we have Auto classes(`AutoModel`).

In [2]:
from datasets import load_dataset

train_data = load_dataset("imdb", split="train")
train_data = train_data.shard(num_shards=4, index=0)

test_data = load_dataset("imdb", split="test")
train_data = train_data.shard(num_shards=4, index=0)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
!pip install transformers huggingface_hub


In [4]:
from transformers import AutoModel , AutoTokenizer
from transformers import AutoModelForSequenceClassification
import torch
import torch_xla.core.xla_model as xm

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
model.to(xm.xla_device())
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, fused=False)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_362/3295043367.py:7: D

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=64)

tokenized_training_data = train_data.map(tokenize_function, batched=True)
tokenized_test_data = test_data.map(tokenize_function, batched=True)

Map:   0%|          | 0/1563 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [6]:
print(tokenized_training_data)

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1563
})


In [7]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir = "./fineTunes",
    eval_strategy ="epoch",
    num_train_epochs= 3,
    learning_rate = 2e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    weight_decay=0.001
)

trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset=tokenized_training_data,
    eval_dataset=tokenized_test_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    optimizers=(optimizer, None)

)

trainer.train()

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,8.600588
2,No log,8.629899
3,0.001950,8.639962


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=588, training_loss=0.0016583112531648391, metrics={'train_runtime': 677.0417, 'train_samples_per_second': 6.948, 'train_steps_per_second': 0.868, 'total_flos': 154215967322880.0, 'train_loss': 0.0016583112531648391, 'epoch': 3.0})

In [8]:
import torch
import torch_xla.core.xla_model as xm

new_data = ["This movie was disappointing","This is the best movie ever"]

new_input = tokenizer(new_data,return_tensors="pt",padding=True, truncation=True,max_length=64)

# Move input tensors to the XLA device
new_input = {k: v.to(xm.xla_device()) for k, v in new_input.items()}

with torch.no_grad():
  output = model(**new_input)

# Get predicted labels from model output
predicted_labels = torch.argmax(output.logits, dim=-1).cpu().numpy()

label_map = {0:"NEGATIVE",1:"POSITIVE"}
for i , predicted_label in enumerate(predicted_labels):
  sentiment = label_map[predicted_label]
  print(f"\nInput Text{i+1}:{new_data[i]}")
  print(f"Predicted label:{sentiment}")

/tmp/ipykernel_362/1212795063.py:9: DeprecationWarning: Use torch_xla.device instead
  new_input = {k: v.to(xm.xla_device()) for k, v in new_input.items()}



Input Text1:This movie was disappointing
Predicted label:NEGATIVE

Input Text2:This is the best movie ever
Predicted label:NEGATIVE


In [13]:
model.to("cpu")
model.save_pretrained("./MyFineTuneModel")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Full Model
#Importing Libraries
from datasets import load_dataset
from transformers import AutoModel , AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
import torch
import torch_xla.core.xla_model as xm


# Preprocessing data
train_data = load_dataset("imdb", split="train")
train_data = train_data.shard(num_shards=4, index=0)

test_data = load_dataset("imdb", split="test")
train_data = train_data.shard(num_shards=4, index=0)

# Model and Tokenizer created
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
model.to(xm.xla_device())
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, fused=False)

# Data Tokenization
def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=64)

tokenized_training_data = train_data.map(tokenize_function, batched=True)
tokenized_test_data = test_data.map(tokenize_function, batched=True)

# Model training

training_args = TrainingArguments(
    output_dir = "./fineTunes",
    eval_strategy ="epoch",
    num_train_epochs= 3,
    learning_rate = 2e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    weight_decay=0.001
)

trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset=tokenized_training_data,
    eval_dataset=tokenized_test_data,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    optimizers=(optimizer, None)

)

trainer.train()

# Model Prediction
new_data = ["This movie was disappointing","This is the best movie ever"]

new_input = tokenizer(new_data,return_tensors="pt",padding=True, truncation=True,max_lengt=64)

with torch.no_grad():
  output = model(**new_input)

label_map = {0:"NEGATIVE",1:"POSITIVE"}
for i , predicted_label in enumerate(predicted_labels):
  sentiment = label_map[predicted_label]
  print(f"\nInput Text{i+1}:{new_data[i]}")
  print(f"Predicted label:{sentiment}")


# Saving Models
model.to("cpu")
model.save_pretrained("./MyFineTuneModel")


In [ ]:
# Model Reutlization
model = AutoModelForSequenceClassification.from_pretrained("my_finetuned_files")
tokenizer = AutoTokenizer.from_pretrained("my_finetuned_files")

## Fine Tunning Approaches

* One approach is Full fine tuning where in we update all the parameters of the model

* Other approach is partial tuning where in we tune only the output layer partn of the model.

* The use case on tuning depends on which model we use.
